# My workbench — YMH303

Everything you need, in one place. Run cells with **Shift+Enter**.

| Cell | What it does |
|---|---|
| **1. Start** | connects to your database (run once each time you open the notebook) |
| **2. Rebuild** | wipes your database and rebuilds it from the files in `db/migrations` |
| **3. Check** | runs the table contract test and shows ✅ / ❌ / ⏳ per table |
| **4. Look** | shows the tables and their columns |
| **5. Try** | your SQL experiments (cells starting with `%%sql`) |

**Your routine:** edit your file in `db/migrations/` → save → run **2** → run **3** → fix what is ❌.

In [ ]:
# 1. START: connect (settings come from .env; no password here)
import os, pathlib, subprocess, sys, re
from dotenv import load_dotenv
from sqlalchemy import create_engine

ROOT = pathlib.Path.cwd()
while not (ROOT / "manage.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
engine = create_engine(
    f"postgresql+psycopg://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ.get('POSTGRES_HOST', 'localhost')}:{os.environ.get('POSTGRES_PORT', '5432')}"
    f"/{os.environ['POSTGRES_DB']}"
)
%load_ext sql
%config SqlMagic.displaylimit = 50
%config SqlMagic.displaycon = False
%sql engine

def rebuild():
    r = subprocess.run([sys.executable, "manage.py", "reset"], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout + r.stderr)

def check():
    r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_schema.py", "-v", "-rs", "--no-header", "--color=no", "-p", "no:cacheprovider"],
                       cwd=ROOT, capture_output=True, text=True)
    out = r.stdout
    for line in out.splitlines():
        m = re.search(r"test_table_matches_report\[(\w+)\] (PASSED|FAILED|SKIPPED)", line)
        if m:
            icon = {"PASSED": "✅", "FAILED": "❌", "SKIPPED": "⏳ file not written yet"}[m.group(2)]
            print(f"{icon}  {m.group(1)}")
    errors = [l.split("AssertionError: ", 1)[1] for l in out.splitlines() if "AssertionError: " in l and l.startswith("E")]
    if errors:
        print("\nTo fix:")
        for e in dict.fromkeys(errors):
            print("  •", e)
    print("\n" + (out.strip().splitlines() or ["no output"])[-1])

print("Connected. Next: run cell 2 (rebuild) and cell 3 (check).")

## 2. Rebuild the database from the migration files

In [ ]:
rebuild()

## 3. Check the tables against the report

In [ ]:
check()

## 4. Look: tables in `core`, then the columns of one table (change `'topic'`)

In [ ]:
%%sql
SELECT table_name FROM information_schema.tables
WHERE table_schema = 'core' ORDER BY table_name;

In [ ]:
%%sql
SELECT column_name, data_type, is_nullable, column_default
FROM information_schema.columns
WHERE table_schema = 'core' AND table_name = 'topic'
ORDER BY ordinal_position;

## 5. Try data
One test per cell. A good row must be accepted; a bad row must be **refused** — the error *is* the test passing.
When finished, run cell 2 again to clean up.

In [ ]:
%%sql
INSERT INTO core.topic (name, acronym) VALUES ('Databases', 'DB');
SELECT * FROM core.topic;

In [ ]:
%%sql
-- must be refused: same name twice (UNIQUE)
INSERT INTO core.topic (name) VALUES ('Databases');